# Exam (Base Notebook)

**Instructions**

- Work independently. No AI or other help was used. 
- Do **not** apply any preprocessing/augmentation beyond normalization provided here.
- Your goal: **produce the best generalization** on the hidden test split while keeping the model efficient.
- You must **compare validation accuracy to test accuracy** in a plot and briefly reflect on any gap.
- You may modify only the sections marked **Your work**. Do not change fixed cells.


## 0. Honor Statement
I, `WRITE YOUR NAME HERE`, assure that I have completed this exam independently and followed all rules.

Shoshe Asanin

 Import the libraries 

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers


TensorFlow: 2.20.0


Loading the MNIST handwritten digit dataset 

In [ ]:
# Load MNIST dataset
# It gives us training images and test images
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Print the shape of the data
print("Training data shape:", X_train_full.shape)
print("Test data shape:", X_test.shape)




 Normalize the image pixel values and reshape the data 

In [ ]:
# Change image values from 0-255 to 0-1
# This makes training easier for the model
X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Add 1 channel 
X_train_full = np.expand_dims(X_train_full, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print("Training shape after preprocessing:", X_train_full.shape)
print("Test shape after preprocessing:", X_test.shape)

 distribution of digit classes in the training dataset

In [ ]:
# Split the original training data
# 80% for training and 20% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Test set shape:", X_test.shape)


check class count

In [ ]:
#  how many samples we have for each digit
unique, counts = np.unique(y_train, return_counts=True)

print("Training class counts:")
for digit, count in zip(unique, counts):
    print(f"Digit {digit}: {count}")

 some images

In [ ]:
# some sample images from the training set
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

function to draw accuracy

In [ ]:
#  training and validation graphs
def plot_curves(history, title="Model"):
    plt.figure(figsize=(12, 4))

    # Accuracy graph
    plt.subplot(1, 2, 1)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.title(f"{title} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss graph
    plt.subplot(1, 2, 2)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{title} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()

## 1. Baseline Model
Check this CNN baseline. Leave as it is. Start fixing it in the next stage.

Check the distribution of digit classes in the training dataset.

In [ ]:

baseline_model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),                    # change image into one long vector
    layers.Dense(10, activation="softmax")  # 10 outputs for digits 0-9
])

# Compile the model
baseline_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
baseline_history = baseline_model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_val, y_val),
    verbose=0
)

# Test the model
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)

print("Baseline model total parameters:", baseline_model.count_params())
print(f"Baseline test accuracy: {baseline_test_acc:.4f}")

# Show graphs
plot_curves(baseline_history, "Baseline Model")

## 2. Learning Curves (Fixed utility)
Plot training vs validation curves to diagnose generalization.

improved CNN model

In [ ]:
# This is the improved model

final_model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),

    # First CNN block
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Second CNN block
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Third CNN layer
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),

    # This reduces the number of parameters
    layers.GlobalAveragePooling2D(),

    # Small dense layer
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),

    # Output layer
    layers.Dense(10, activation="softmax")
])

# Compile the model
final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
final_history = final_model.fit(
    X_train,
    y_train,
    epochs=12,
    validation_data=(X_val, y_val),
    verbose=0
)

# Test the final model
final_test_loss, final_test_acc = final_model.evaluate(X_test, y_test, verbose=0)

print("Final model total parameters:", final_model.count_params())
print(f"Final test accuracy: {final_test_acc:.4f}")

# Show graphs
plot_curves(final_history, "Final CNN Model")

 saving model

In [ ]:
# Save the model as .keras file
final_model.save("mnist_final_model.keras")

print("Model saved successfully as mnist_final_model.keras")

 summary

In [ ]:
# Print model layers and parameter counts
final_model.summary()

## 3. Validation vs Test Accuracy (Required)
Evaluate on the held-out test set and overlay the test accuracy on the validation curve. Then write a short reflection (3–5 sentences) on the generalization gap.

In [ ]:
# how many test images were predicted correctly
# There are 10,000 test images in MNIST
correct_predictions = int(round(final_test_acc * len(X_test)))

# Count the total number of model parameters
total_params = final_model.count_params()

# Reward calculation based on the exam question
reward = 0

if correct_predictions <= 5000:
    reward = correct_predictions * 100
elif correct_predictions <= 6000:
    reward = (5000 * 100) + ((correct_predictions - 5000) * 200)
else:
    reward = (5000 * 100) + (1000 * 200) + ((correct_predictions - 6000) * 1000)

# Final score
net_score = reward - total_params

print("Correct predictions:", correct_predictions)
print("Reward:", reward)
print("Parameter penalty:", total_params)
print("Net Score:", net_score)

predection model

In [ ]:
# Predict labels for the test images
pred_probs = final_model.predict(X_test, verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)

# first 10 predictions
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].squeeze(), cmap="gray")
    plt.title(f"True: {y_test[i]}\nPred: {pred_labels[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

reflexion

The validation accuracy and test accuracy are close, so the model seems to work well on unseen data. A small difference is normal because the validation set is used during training, but the test set is only used at the end. If the difference becomes too big, it can mean overfitting. In that case, I could reduce the model size, use dropout, or train for fewer epochs. In this work, the data was split into training, validation, and test sets so the model could be trained, checked, and finally evaluated fairly.

### Reflection (Your text)
- Explain the gap between validation and test accuracy.
- What might cause it, and what would you try next to reduce it?
- How much of the dataset do you use for training, validation, and testing? Explain your answer with the % calculation.
- How many images are there in the original dataset?
- How big are the images?
- What observations can you tell about the original data?  
- What are possible labels for the images?
- What are the steps when making models?
- <b>Make sure you explain every step well!<b>
- <b>Make sure your model is as optimized as possible!<b>

## 4. Improved Model (Your work)
Build a better model, train it here as many times as you feel like and when you are pleased with it, title it `final_model`.  Report every try individually, not only the final version.

Explain every step with markdown text and code comments.   

Unrunnable code is not checked. 